# 01 · Generador de dataset sintético (Fase 1a)
**Horizonte Clínico / SOC-IA · Unidad piloto: Cardiovascular, Hospital San José de Melipilla**

Qué hace: simula ~12 semanas de operación de 4 recintos (consulta, ecocardiografía, Holter, prueba de esfuerzo) y guarda `ceaa_dataset.csv`.
Cada fila es **un bloque horario de un recinto**: si se realizó la atención o no, y por qué no.

Cómo se reparametriza: edita **`parametros.yaml`** (misma carpeta) con los datos de Melipilla y ejecuta todo de nuevo. No hay que tocar código.

Ejecutar: *Entorno de ejecución → Ejecutar todo*. La primera vez Colab pide autorizar el acceso a Drive.

In [4]:
import os, json
import numpy as np, pandas as pd, yaml

try:                                    # En Colab: monta Drive
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/HORIZONTE_CLINICO'
except ImportError:                     # Fuera de Colab (pruebas locales)
    BASE = os.environ.get('HC_BASE', os.path.abspath('..'))

DIR = f'{BASE}/02_fase1_datos'
os.makedirs(DIR, exist_ok=True)
with open(f'{DIR}/parametros.yaml', encoding='utf-8') as f:
    P = yaml.safe_load(f)
print('Carpeta de trabajo:', DIR)
print('Unidad piloto     :', P['meta']['unidad'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta de trabajo: /content/drive/MyDrive/HORIZONTE_CLINICO/02_fase1_datos
Unidad piloto     : Cardiovascular


## 1. ¿Con qué datos se está generando? (procedencia de cada parámetro)
Si algún bloque sigue como `ESTANDAR_PROVISORIO`, el dataset es **provisorio** y la memoria debe decirlo.

In [5]:
fuentes = P['fuentes']
display(pd.DataFrame({'bloque de parámetros': list(fuentes), 'fuente': list(fuentes.values())}))
PROVISORIO = any(str(v).startswith('ESTANDAR') for v in fuentes.values())
if PROVISORIO:
    print('AVISO: hay parámetros ESTÁNDAR. El dataset es PROVISORIO: no son hallazgos reales de Melipilla.')
else:
    print('Todos los bloques provienen de Melipilla (exacto o estimado).')

,bloque de parámetros,fuente
0,agenda,ESTANDAR_PROVISORIO
1,recintos,ESTANDAR_PROVISORIO
2,dotacion,ESTANDAR_PROVISORIO
3,tasas,ESTANDAR_PROVISORIO
4,demanda,ESTANDAR_PROVISORIO
5,duracion,ESTANDAR_PROVISORIO


AVISO: hay parámetros ESTÁNDAR. El dataset es PROVISORIO: no son hallazgos reales de Melipilla.


## 2. Bloques horarios y profesionales

In [6]:
rng = np.random.default_rng(P['meta']['semilla'])
A, T, D, DEM = P['agenda'], P['tasas'], P['duracion'], P['demanda']
DIAS = ['Lun', 'Mar', 'Mie', 'Jue', 'Vie']

def hhmm(s):
    h, m = map(int, s.split(':')); return h*60 + m

ini, fin = hhmm(A['inicio']), hhmm(A['termino'])
col_i, col_f = hhmm(A['colacion'][0]), hhmm(A['colacion'][1])
bloques, t = [], ini
while t + A['bloque_min'] <= fin:
    if not (col_i <= t < col_f):          # se omiten los bloques de colación
        bloques.append(t)
    t += A['bloque_min']

horas_dia = {'manana': sum(b < col_i for b in bloques) * A['bloque_min']/60,
             'completa': len(bloques) * A['bloque_min']/60}
print(f"{len(bloques)} bloques por día ({[f'{b//60:02d}:{b%60:02d}' for b in bloques][:3]} ... ) · "
      f"jornada mañana = {horas_dia['manana']} h, jornada completa = {horas_dia['completa']} h")

profs = []
for tipo, d in P['dotacion'].items():
    for k in range(d['n']):
        solo_m = k < d['solo_manana']
        hd = horas_dia['manana'] if solo_m else horas_dia['completa']
        n_dias = int(min(len(A['dias']), max(1, round(d['horas_sem'] / hd))))
        dias = sorted(int(x) for x in rng.choice(A['dias'], size=n_dias, replace=False))
        profs.append({'id': f"{d['codigo']}{k+1:02d}", 'tipo': tipo, 'solo_manana': solo_m, 'dias': dias})
display(pd.DataFrame(profs))

16 bloques por día (['08:00', '08:30', '09:00'] ... ) · jornada mañana = 5.0 h, jornada completa = 8.0 h


,id,tipo,solo_manana,dias
0,CAR01,cardiologo,True,"[0, 2, 3, 4]"
1,CAR02,cardiologo,True,"[0, 1, 2, 3]"
2,CAR03,cardiologo,False,"[2, 3, 4]"
3,TMD01,tecnologo_medico,True,"[0, 1, 2, 3, 4]"
4,TMD02,tecnologo_medico,False,"[1, 2, 3, 4]"
5,TEC01,tecnico,True,"[0, 1, 2, 3, 4]"
6,TEC02,tecnico,False,"[0, 1, 2, 3, 4]"
7,ENF01,enfermera,True,"[0, 1, 2, 3, 4]"
8,ENF02,enfermera,False,"[0, 1, 2, 3, 4]"


## 3. Ausencias (licencias, pasantías, ausentismo)
Las licencias duran varios días seguidos (por eso aparecen "dos bajas simultáneas"); pasantías y ausentismo se sortean por día.

In [7]:
semanas = P['meta']['semanas']
fecha0 = pd.Timestamp(P['meta']['fecha_inicio'])
C = T['licencia_persistencia']
p_inicio_lic = T['licencia'] * (1 - C) / (1 - T['licencia'])   # mantiene la tasa de licencia objetivo

ausencia = {}
for p in profs:
    en_lic = False
    for w in range(semanas):
        for d in A['dias']:
            en_lic = (rng.random() < C) if en_lic else (rng.random() < p_inicio_lic)
            causa = 'licencia' if en_lic else None
            if causa is None:
                u = rng.random()
                if u < T['pasantia']: causa = 'pasantia'
                elif u < T['pasantia'] + T['ausentismo']: causa = 'ausentismo'
            ausencia[(p['id'], w, d)] = causa

tasa_real = pd.Series(list(ausencia.values())).fillna('presente').value_counts(normalize=True).round(3)
print('Proporción de profesional-días:'); print(tasa_real.to_string())

Proporción de profesional-días:
presente      0.796
licencia      0.107
pasantia      0.065
ausentismo    0.031


## 4. Operación bloque a bloque
Para cada recinto y bloque: ¿había un profesional programado y presente? ¿había paciente? ¿llegó? ¿se reprogramó?
`prof_sin_box` cuenta profesionales presentes que **no tienen recinto** en ese bloque (hay un solo recinto por tipo): es la asincronía profesional–box.

In [8]:
rows = []
for w in range(semanas):
    for d in A['dias']:
        fecha = fecha0 + pd.Timedelta(weeks=w, days=d)
        for b in bloques:
            jornada = 'manana' if b < col_i else 'tarde'
            for r in P['recintos']:
                tp = r['prof_tipo']
                prog = [p for p in profs if p['tipo'] == tp and d in p['dias'] and not (p['solo_manana'] and jornada == 'tarde')]
                disp = [p for p in prog if ausencia[(p['id'], w, d)] is None]
                prof_id, estado, causa, dur_real = np.nan, 'no_realizado', None, np.nan

                if not prog:
                    hay_manana = any(p['tipo'] == tp and d in p['dias'] and p['solo_manana'] for p in profs)
                    causa = 'concentracion_manana' if (jornada == 'tarde' and hay_manana) else 'sin_profesional_programado'
                elif not disp:
                    causa = ausencia[(prog[0]['id'], w, d)]
                else:
                    prof_id = disp[rng.integers(len(disp))]['id']
                    if rng.random() > DEM[jornada]:
                        causa = 'sin_paciente_agendado'
                    elif rng.random() < T['no_show_paciente']:
                        causa = 'no_show_paciente'
                    elif rng.random() < T['reprogramacion_institucional']:
                        estado, causa = 'reprogramado', 'reprogramacion_institucional'
                    else:
                        estado, causa = 'realizado', 'ninguna'
                        dur_real = int(np.clip(round(r['dur_agendada'] * rng.normal(D['media_factor'], D['sd'])), 5, 2*A['bloque_min']))

                rows.append({
                    'fecha': fecha.date(), 'semana': w+1, 'dia_semana': DIAS[d],
                    'bloque_horario': f'{b//60:02d}:{b%60:02d}', 'jornada': jornada,
                    'ID_recinto': r['id'], 'tipo_recinto': r['tipo'],
                    'ID_profesional': prof_id, 'tipo_profesional': tp,
                    'codigo_prestacion': r['codigo'],
                    'duracion_agendada_min': r['dur_agendada'], 'duracion_real_min': dur_real,
                    'estado_bloque': estado, 'causa_no_uso': causa,
                    'prof_disponibles': len(disp), 'prof_sin_box': max(len(disp) - 1, 0),
                })
df = pd.DataFrame(rows)
print(df.shape)
display(df.head(8))

(3840, 16)


,fecha,semana,dia_semana,bloque_horario,jornada,ID_recinto,tipo_recinto,ID_profesional,tipo_profesional,codigo_prestacion,duracion_agendada_min,duracion_real_min,estado_bloque,causa_no_uso,prof_disponibles,prof_sin_box
0,2026-03-02,1,Lun,08:00,manana,R01,consulta_cardio,CAR01,cardiologo,PRE-CONS,20,NaN,no_realizado,sin_paciente_agendado,2,1
1,2026-03-02,1,Lun,08:00,manana,R02,ecocardiografia,TMD01,tecnologo_medico,PRE-ECO,30,32.0,realizado,ninguna,1,0
2,2026-03-02,1,Lun,08:00,manana,R03,holter,TEC02,tecnico,PRE-HOL,30,41.0,realizado,ninguna,2,1
3,2026-03-02,1,Lun,08:00,manana,R04,prueba_esfuerzo,ENF02,enfermera,PRE-ESF,30,NaN,reprogramado,reprogramacion_institucional,2,1
4,2026-03-02,1,Lun,08:30,manana,R01,consulta_cardio,CAR02,cardiologo,PRE-CONS,20,21.0,realizado,ninguna,2,1
5,2026-03-02,1,Lun,08:30,manana,R02,ecocardiografia,TMD01,tecnologo_medico,PRE-ECO,30,NaN,no_realizado,sin_paciente_agendado,1,0
6,2026-03-02,1,Lun,08:30,manana,R03,holter,TEC02,tecnico,PRE-HOL,30,35.0,realizado,ninguna,2,1
7,2026-03-02,1,Lun,08:30,manana,R04,prueba_esfuerzo,ENF02,enfermera,PRE-ESF,30,38.0,realizado,ninguna,2,1


## 5. Chequeos de calidad (si alguno falla, la celda se detiene)

In [9]:
assert 3000 <= len(df) <= 5000, f'Filas fuera del rango 3.000–5.000: {len(df)}'
assert df.duplicated(['fecha', 'bloque_horario', 'ID_recinto']).sum() == 0, 'Hay bloques duplicados'
assert df.loc[df.estado_bloque == 'realizado', 'duracion_real_min'].notna().all()
assert df.loc[df.estado_bloque != 'realizado', 'causa_no_uso'].notna().all()
print('Filas:', len(df), '· recintos:', df.ID_recinto.nunique(), '· profesionales:', df.ID_profesional.nunique())

ocup = (df.estado_bloque == 'realizado')
print('\nOcupación global       :', f'{ocup.mean():.1%}')
print('Ocupación por jornada  :'); print(ocup.groupby(df.jornada).mean().map('{:.1%}'.format).to_string())
print('Ocupación por recinto  :'); print(ocup.groupby(df.tipo_recinto).mean().map('{:.1%}'.format).to_string())
print('\nEstados:'); print(df.estado_bloque.value_counts().to_string())

Filas: 3840 · recintos: 4 · profesionales: 9

Ocupación global       : 63.5%
Ocupación por jornada  :
jornada
manana    69.7%
tarde     53.1%
Ocupación por recinto  :
tipo_recinto
consulta_cardio    59.1%
ecocardiografia    61.1%
holter             61.7%
prueba_esfuerzo    72.0%

Estados:
estado_bloque
realizado       2437
no_realizado    1288
reprogramado     115


## 6. Guardar dataset, diccionario y metadatos

In [10]:
df.to_csv(f'{DIR}/ceaa_dataset.csv', index=False, encoding='utf-8')

dic = pd.DataFrame([
 ('fecha', 'Fecha del bloque (dato simulado)'), ('semana', 'N.º de semana simulada (1..N)'),
 ('dia_semana', 'Lun a Vie'), ('bloque_horario', 'Hora de inicio del bloque (hh:mm)'),
 ('jornada', 'manana (antes de la colación) o tarde'),
 ('ID_recinto', 'Identificador interno del recinto'), ('tipo_recinto', 'consulta_cardio, ecocardiografia, holter, prueba_esfuerzo'),
 ('ID_profesional', 'Identificador pseudónimo del profesional asignado al bloque (vacío si no había)'),
 ('tipo_profesional', 'Tipo de profesional requerido por el recinto'),
 ('codigo_prestacion', 'Código INTERNO provisorio de la prestación'),
 ('duracion_agendada_min', 'Duración agendada de la prestación (min)'),
 ('duracion_real_min', 'Duración real (min); vacío si el bloque no se realizó'),
 ('estado_bloque', 'realizado, no_realizado o reprogramado'),
 ('causa_no_uso', 'ninguna, licencia, pasantia, ausentismo, concentracion_manana, sin_profesional_programado, sin_paciente_agendado, no_show_paciente, reprogramacion_institucional'),
 ('prof_disponibles', 'Profesionales del tipo requerido presentes y programados en ese bloque'),
 ('prof_sin_box', 'Profesionales presentes que no tienen recinto en ese bloque (prof_disponibles - 1, mínimo 0)'),
], columns=['columna', 'descripcion'])
dic.to_csv(f'{DIR}/diccionario_datos.csv', index=False, encoding='utf-8')

meta = {'generado': pd.Timestamp.now().isoformat(timespec='seconds'), 'filas': int(len(df)),
        'provisorio': bool(PROVISORIO), 'fuentes': fuentes, 'semilla': P['meta']['semilla'],
        'unidad': P['meta']['unidad']}
with open(f'{DIR}/metadata_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('Guardado en', DIR); print(sorted(os.listdir(DIR)))

Guardado en /content/drive/MyDrive/HORIZONTE_CLINICO/02_fase1_datos
['01_dataset_CEAA.ipynb', '02_EDA_CEAA.ipynb', 'ceaa_dataset.csv', 'diccionario_datos.csv', 'figuras', 'metadata_dataset.json', 'parametros.yaml']


**Siguiente paso:** abrir `02_EDA_CEAA.ipynb`.
Cuando lleguen los datos de Melipilla: editar `parametros.yaml` (valores y fuentes), volver a ejecutar este cuaderno y luego el 02.